# Section 2. Diff-CAM

The two class maps are nearly identical after alignment, in lesion r 0.90.
The natural response is to subtract them and look at what remains.

**Shelley's observation.** Roughly half of the difference mass sits outside
the lesion. Her proposal was to renormalise inside the lesion so the
difference is measured where the diagnosis is made.

Four tests.

| # | question |
|---|---|
| 1 | Where does the difference mass actually sit |
| 2 | Does renormalising inside the lesion change anything |
| 3 | Is the construction correct |
| 4 | Does it hold across model families |

In [2]:
import pandas as pd
from sklearn.metrics import roc_auc_score
from scipy.ndimage import distance_transform_edt, center_of_mass
from scipy.stats import spearmanr

qcp = pd.read_csv(REPO / "results" / "annotation_qc" / GEOMETRY /
                  "annotation_qc_manifest.csv").query("qc_pass")

def mel_union(iid):
    sub = qcp[(qcp.Image_ID == iid) & (qcp.label_group == "MEL")]
    if not len(sub):
        return None
    acc = None
    for p in sub.mask_path:
        m = np.array(Image.open(p).convert("L")) > 127
        acc = m if acc is None else (acc | m)
    return mask_to_cam_grid_geom(acc, GEOMETRY) >= 0.5

HUM = {i: mel_union(i) for i in eval_df.image_id}
HUM = {k: v for k, v in HUM.items()
       if v is not None and LESION[k].sum() >= 8
       and 0 < v[LESION[k]].sum() < LESION[k].sum()}

def overlay(ax, rgb, cam, thresh=0.28, cmap="turbo", gamma=0.8):
    c = norm01(np.asarray(cam, np.float32))
    u = cv2.resize(c, (CROP_SIZE, CROP_SIZE), interpolation=cv2.INTER_CUBIC)
    al = np.clip((u - thresh) / (1 - thresh), 0, 1) ** gamma * 0.80
    ax.imshow(rgb)
    ax.imshow(u, cmap=cmap, alpha=al, vmin=0, vmax=1, interpolation="bilinear")
    ax.set_xticks([]); ax.set_yticks([])

def overlay_signed(ax, rgb, cam, thresh=0.12):
    c = np.asarray(cam, np.float32)
    v = np.percentile(np.abs(c), 98) + 1e-8
    u = cv2.resize(c, (CROP_SIZE, CROP_SIZE), interpolation=cv2.INTER_CUBIC)
    al = np.clip((np.abs(u) / v - thresh) / (1 - thresh), 0, 1) ** 0.8 * 0.85
    ax.imshow(rgb)
    ax.imshow(u, cmap="RdBu_r", alpha=al, vmin=-v, vmax=v,
              interpolation="bilinear")
    ax.set_xticks([]); ax.set_yticks([])

auc = lambda y, v: float(roc_auc_score(y, v)) if np.std(v) > 1e-9 else np.nan
print(f"{len(HUM)} images with a usable in lesion MEL union")

NameError: name 'REPO' is not defined